[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

# Shape Inference — Apply

Hands-on exercises for ONNX shape inference.  You will build models, run `infer_shapes()`,
verify output formulas manually, handle symbolic/dynamic dimensions, debug mismatches,
and implement shape inference for a custom operator.

## Table of Contents

| # | Exercise | Objective |
|---|----------|----------|
| 1 | [Setup](#1) | Install and import dependencies |
| 2 | [Basic Shape Inference](#2) | Run inference on a linear model |
| 3 | [Inspect Inferred Types](#3) | Read TypeProto from value_info |
| 4 | [Conv Shape Formula](#4) | Verify the output size formula |
| 5 | [Multi-Layer CNN](#5) | Track shapes through convolutions |
| 6 | [Broadcasting](#6) | Infer shapes for broadcast ops |
| 7 | [Symbolic Dimensions](#7) | Dynamic batch and sequence |
| 8 | [Partial Inference](#8) | Data-dependent shape ops |
| 9 | [Debug Shape Mismatch](#9) | Find and fix errors |
| 10 | [Custom Op Shape](#10) | Manual annotation for unknown ops |

<a id="1"></a>
## Exercise 1 — Setup

In [ ]:
# !pip install onnx onnxruntime numpy matplotlib --quiet

import numpy as np
import onnx
from onnx import helper, TensorProto, shape_inference, checker
print(f"ONNX version: {onnx.__version__}")

In [ ]:
def get_shapes(model_proto):
    """Run shape inference and return a dict of name -> shape list."""
    inferred = shape_inference.infer_shapes(model_proto)
    shapes = {}
    for vi in list(inferred.graph.value_info) + list(inferred.graph.output):
        tt = vi.type.tensor_type
        if tt.HasField("shape"):
            dims = [d.dim_param if d.dim_param else d.dim_value for d in tt.shape.dim]
            shapes[vi.name] = dims
    return shapes

def show_shapes(model_proto):
    """Print all inferred shapes."""
    shapes = get_shapes(model_proto)
    for name, dims in shapes.items():
        print(f"  {name}: {dims}")
    return shapes

print("Utility functions defined.")

<a id="2"></a>
## Exercise 2 — Basic Shape Inference

Build a two-layer linear model (MatMul + Relu + MatMul) and run shape inference.
Verify that intermediate shapes match your expectations.

In [ ]:
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [8, 784])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

W1 = helper.make_tensor("W1", TensorProto.FLOAT, [784, 128], np.zeros((784, 128), dtype=np.float32).flatten())
W2 = helper.make_tensor("W2", TensorProto.FLOAT, [128, 10], np.zeros((128, 10), dtype=np.float32).flatten())

nodes = [
    helper.make_node("MatMul", ["X", "W1"], ["H"]),
    helper.make_node("Relu", ["H"], ["R"]),
    helper.make_node("MatMul", ["R", "W2"], ["Y"]),
]

graph = helper.make_graph(nodes, "two_layer", [X], [Y], initializer=[W1, W2])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

print("Inferred shapes:")
shapes = show_shapes(model)

# Verify manually
assert shapes["H"] == [8, 128], f"Expected [8, 128] but got {shapes['H']}"
assert shapes["R"] == [8, 128], f"Expected [8, 128] but got {shapes['R']}"
assert shapes["Y"] == [8, 10], f"Expected [8, 10] but got {shapes['Y']}"
print("\nAll shapes match expectations!")

<a id="3"></a>
## Exercise 3 — Inspect Inferred Types

After inference, each `ValueInfoProto` carries both element type and shape.  Extract and
display the full type information for every intermediate tensor.

In [ ]:
inferred = shape_inference.infer_shapes(model)

print(f"{'Name':<10s}  {'Elem Type':<12s}  {'Shape':<20s}  {'Rank':>4s}  {'Numel':>10s}")
print("-" * 62)

all_infos = list(inferred.graph.input) + list(inferred.graph.value_info) + list(inferred.graph.output)

for vi in all_infos:
    tt = vi.type.tensor_type
    dtype_name = TensorProto.DataType.Name(tt.elem_type)
    if tt.HasField("shape"):
        dims = [d.dim_param if d.dim_param else d.dim_value for d in tt.shape.dim]
        rank = len(dims)
        concrete_dims = [d for d in dims if isinstance(d, int) and d > 0]
        numel = int(np.prod(concrete_dims)) if concrete_dims else "?"
    else:
        dims = "unknown"
        rank = "?"
        numel = "?"
    print(f"{vi.name:<10s}  {dtype_name:<12s}  {str(dims):<20s}  {str(rank):>4s}  {str(numel):>10s}")

<a id="4"></a>
## Exercise 4 — Conv Shape Formula Verification

Build Conv nodes with different configurations and verify the output shape formula:

$$H_{out} = \left\lfloor \frac{H_{in} + 2p - d(k-1) - 1}{s} \right\rfloor + 1$$

In [ ]:
def verify_conv_shape(h_in, w_in, c_in, c_out, k, p, s, d=1):
    """Build a Conv node, infer shapes, and compare to formula."""
    # Formula
    h_formula = (h_in + 2 * p - d * (k - 1) - 1) // s + 1
    w_formula = (w_in + 2 * p - d * (k - 1) - 1) // s + 1

    # Build model
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, c_in, h_in, w_in])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
    W_init = np.zeros((c_out, c_in, k, k), dtype=np.float32)
    W_tensor = helper.make_tensor("W", TensorProto.FLOAT, list(W_init.shape), W_init.flatten())

    conv = helper.make_node("Conv", ["X", "W"], ["Y"],
                            kernel_shape=[k, k], pads=[p, p, p, p],
                            strides=[s, s], dilations=[d, d])
    graph = helper.make_graph([conv], "conv_verify", [X], [Y], initializer=[W_tensor])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

    inferred = shape_inference.infer_shapes(model)
    out_dims = [dim.dim_value for dim in inferred.graph.output[0].type.tensor_type.shape.dim]
    h_inferred, w_inferred = out_dims[2], out_dims[3]

    match = (h_formula == h_inferred and w_formula == w_inferred)
    return h_formula, w_formula, h_inferred, w_inferred, match

test_configs = [
    {"h_in": 28, "w_in": 28, "c_in": 1,  "c_out": 16, "k": 5, "p": 0, "s": 1, "d": 1},
    {"h_in": 32, "w_in": 32, "c_in": 3,  "c_out": 64, "k": 3, "p": 1, "s": 1, "d": 1},
    {"h_in": 64, "w_in": 64, "c_in": 64, "c_out": 128, "k": 3, "p": 0, "s": 2, "d": 1},
    {"h_in": 56, "w_in": 56, "c_in": 64, "c_out": 64,  "k": 3, "p": 2, "s": 1, "d": 2},
    {"h_in": 224, "w_in": 224, "c_in": 3, "c_out": 64, "k": 7, "p": 3, "s": 2, "d": 1},
]

print(f"{'Config':<35s}  {'Formula':>10s}  {'Inferred':>10s}  {'Match':>5s}")
print("-" * 68)
for cfg in test_configs:
    h_f, w_f, h_i, w_i, ok = verify_conv_shape(**cfg)
    label = f"[{cfg['h_in']}x{cfg['w_in']}] k={cfg['k']} p={cfg['p']} s={cfg['s']} d={cfg['d']}"
    print(f"{label:<35s}  {h_f:>4d}x{w_f:<4d}  {h_i:>4d}x{w_i:<4d}   {'OK' if ok else 'FAIL'}")

<a id="5"></a>
## Exercise 5 — Multi-Layer CNN Shape Tracking

Build a small CNN (Conv -> Relu -> Pool -> Conv -> Relu -> Pool -> Flatten -> FC)
and visualize how spatial dimensions shrink while channel count grows.

In [ ]:
import matplotlib.pyplot as plt

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 1, 28, 28])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

inits = [
    helper.make_tensor("W1", TensorProto.FLOAT, [16, 1, 5, 5],
                       np.zeros((16, 1, 5, 5), dtype=np.float32).flatten()),
    helper.make_tensor("W2", TensorProto.FLOAT, [32, 16, 3, 3],
                       np.zeros((32, 16, 3, 3), dtype=np.float32).flatten()),
    helper.make_tensor("W3", TensorProto.FLOAT, [10, 800],
                       np.zeros((10, 800), dtype=np.float32).flatten()),
]

shape_target = np.array([1, 800], dtype=np.int64)
nodes = [
    helper.make_node("Conv", ["X", "W1"], ["C1"], kernel_shape=[5, 5]),
    helper.make_node("Relu", ["C1"], ["R1"]),
    helper.make_node("MaxPool", ["R1"], ["P1"], kernel_shape=[2, 2], strides=[2, 2]),
    helper.make_node("Conv", ["P1", "W2"], ["C2"], kernel_shape=[3, 3]),
    helper.make_node("Relu", ["C2"], ["R2"]),
    helper.make_node("MaxPool", ["R2"], ["P2"], kernel_shape=[2, 2], strides=[2, 2]),
    helper.make_node("Constant", [], ["shape_t"],
                     value=helper.make_tensor("sv", TensorProto.INT64, [2], shape_target)),
    helper.make_node("Reshape", ["P2", "shape_t"], ["flat"]),
    helper.make_node("Gemm", ["flat", "W3"], ["Y"], transB=1),
]

graph = helper.make_graph(nodes, "cnn", [X], [Y], initializer=inits)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
checker.check_model(model)

shapes = get_shapes(model)
print("CNN shape propagation:")
print(f"  X:    [1, 1, 28, 28]")
for name, dims in shapes.items():
    print(f"  {name}: {dims}")

# Visualize
conv_layers = [("Input", 1, 28), ("Conv1", 16, 24), ("Pool1", 16, 12),
               ("Conv2", 32, 10), ("Pool2", 32, 5)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
names = [l[0] for l in conv_layers]
channels = [l[1] for l in conv_layers]
spatials = [l[2] for l in conv_layers]

ax1.bar(names, spatials, color="steelblue", alpha=0.8)
for i, v in enumerate(spatials):
    ax1.text(i, v + 0.5, str(v), ha="center", fontweight="bold")
ax1.set_title("Spatial Size")
ax1.set_ylabel("Height/Width")

ax2.bar(names, channels, color="coral", alpha=0.8)
for i, v in enumerate(channels):
    ax2.text(i, v + 0.5, str(v), ha="center", fontweight="bold")
ax2.set_title("Channel Count")
ax2.set_ylabel("Channels")

plt.tight_layout()
plt.show()

<a id="6"></a>
## Exercise 6 — Broadcasting Shape Inference

Test broadcasting rules for element-wise operations.  Verify that:
- Dimension alignment works from the right.
- Size-1 dimensions expand.
- Incompatible dimensions are caught.

In [ ]:
def test_broadcast(shape_a, shape_b, op="Add"):
    A = helper.make_tensor_value_info("A", TensorProto.FLOAT, shape_a)
    B = helper.make_tensor_value_info("B", TensorProto.FLOAT, shape_b)
    C = helper.make_tensor_value_info("C", TensorProto.FLOAT, None)
    graph = helper.make_graph(
        [helper.make_node(op, ["A", "B"], ["C"])],
        "bcast", [A, B], [C]
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    try:
        shapes = get_shapes(model)
        return shapes.get("C", "unknown")
    except Exception as e:
        return f"Error: {e}"

cases = [
    ([4, 256],      [256],          "Bias add"),
    ([8, 1, 6, 1],  [7, 1, 5],      "Multi-dim"),
    (["N", 3, 224, 224], [1, 3, 1, 1], "Channel-wise"),
    ([4, 1, 3],     [4, 5, 3],      "Middle dim"),
    ([10],          [1],            "Scalar-like"),
]

for sa, sb, label in cases:
    result = test_broadcast(sa, sb)
    print(f"  {label:<16s}  A={str(sa):<22s}  B={str(sb):<16s}  -> C={result}")

<a id="7"></a>
## Exercise 7 — Symbolic Dimensions in Dynamic Models

Build a transformer-style model with symbolic batch and sequence length.
Verify that symbols propagate correctly through MatMul, Add, and Softmax.

In [ ]:
# Transformer-style: Q*K^T / sqrt(d) -> softmax -> V
d_model = 64

Q = helper.make_tensor_value_info("Q", TensorProto.FLOAT, ["batch", "seq", d_model])
K = helper.make_tensor_value_info("K", TensorProto.FLOAT, ["batch", "seq", d_model])
V = helper.make_tensor_value_info("V", TensorProto.FLOAT, ["batch", "seq", d_model])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

perm = [0, 2, 1]
nodes = [
    helper.make_node("Transpose", ["K"], ["Kt"], perm=perm),
    helper.make_node("MatMul", ["Q", "Kt"], ["QK"]),
    helper.make_node("Softmax", ["QK"], ["attn"], axis=-1),
    helper.make_node("MatMul", ["attn", "V"], ["Y"]),
]

graph = helper.make_graph(nodes, "attention", [Q, K, V], [Y])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

print("Attention shape propagation with symbolic dims:")
shapes = show_shapes(model)

# Verify: Kt should be [batch, d_model, seq], QK should be [batch, seq, seq]
assert "batch" in str(shapes.get("Kt", [])), "batch symbol should propagate to Kt"
print("\nSymbolic dimensions propagated correctly through attention.")

<a id="8"></a>
## Exercise 8 — Partial Inference

Some ops produce data-dependent output shapes (e.g., `NonZero`).
Observe how inference handles these cases.

In [ ]:
# NonZero: output shape depends on data values, not just input shape
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [3, 4])
Y = helper.make_tensor_value_info("indices", TensorProto.INT64, None)

graph = helper.make_graph(
    [helper.make_node("NonZero", ["X"], ["indices"])],
    "partial", [X], [Y]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
inferred = shape_inference.infer_shapes(model)

out_type = inferred.graph.output[0].type.tensor_type
if out_type.HasField("shape"):
    dims = []
    for d in out_type.shape.dim:
        if d.dim_param:
            dims.append(d.dim_param)
        elif d.dim_value > 0:
            dims.append(d.dim_value)
        else:
            dims.append("?")
    print(f"NonZero output shape: {dims}")
    print("The second dimension is unknown because it depends on how many non-zero elements exist.")
else:
    print("NonZero output: shape completely unknown (partial inference).")

# Contrast with Transpose — always fully inferable
X2 = helper.make_tensor_value_info("X2", TensorProto.FLOAT, [3, 4, 5])
Y2 = helper.make_tensor_value_info("Y2", TensorProto.FLOAT, None)
graph2 = helper.make_graph(
    [helper.make_node("Transpose", ["X2"], ["Y2"], perm=[2, 0, 1])],
    "full_infer", [X2], [Y2]
)
model2 = helper.make_model(graph2, opset_imports=[helper.make_opsetid("", 17)])
shapes2 = get_shapes(model2)
print(f"\nTranspose [3,4,5] with perm=[2,0,1] -> {shapes2['Y2']}")
print("Transpose is always fully inferable because perm is an attribute.")

<a id="9"></a>
## Exercise 9 — Debug Shape Mismatch

Intentionally create models with shape problems, then diagnose and fix them.

In [ ]:
# Bug 1: MatMul inner dimension mismatch
print("=== Bug 1: MatMul inner dimension mismatch ===")
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [4, 100])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
W_bad = helper.make_tensor("W", TensorProto.FLOAT, [50, 10],
                           np.zeros((50, 10), dtype=np.float32).flatten())

graph = helper.make_graph(
    [helper.make_node("MatMul", ["X", "W"], ["Y"])],
    "bad1", [X], [Y], initializer=[W_bad]
)
model_bad = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

shapes = get_shapes(model_bad)
if shapes.get("Y"):
    print(f"  Inferred Y shape: {shapes['Y']} — but this is wrong!")
    print(f"  X is [4, 100] and W is [50, 10]: inner dims 100 != 50")
else:
    print("  Shape inference could not determine output — dimension mismatch.")

# Fix
print("\n  Fix: Change W to [100, 10]")
W_fix = helper.make_tensor("W", TensorProto.FLOAT, [100, 10],
                           np.zeros((100, 10), dtype=np.float32).flatten())
graph_fix = helper.make_graph(
    [helper.make_node("MatMul", ["X", "W"], ["Y"])],
    "fix1", [X], [Y], initializer=[W_fix]
)
model_fix = helper.make_model(graph_fix, opset_imports=[helper.make_opsetid("", 17)])
shapes_fix = get_shapes(model_fix)
print(f"  Fixed Y shape: {shapes_fix['Y']}")

# Bug 2: Missing bias shape for broadcasting
print("\n=== Bug 2: Concat with mismatched non-axis dims ===")
A = helper.make_tensor_value_info("A", TensorProto.FLOAT, [4, 64])
B = helper.make_tensor_value_info("B", TensorProto.FLOAT, [4, 128])
C = helper.make_tensor_value_info("C", TensorProto.FLOAT, None)

graph_cat = helper.make_graph(
    [helper.make_node("Concat", ["A", "B"], ["C"], axis=1)],
    "concat_ok", [A, B], [C]
)
model_cat = helper.make_model(graph_cat, opset_imports=[helper.make_opsetid("", 17)])
shapes_cat = get_shapes(model_cat)
print(f"  Concat [4,64] + [4,128] on axis=1 -> {shapes_cat['C']}")
print(f"  Rule: axis dim = 64 + 128 = 192; non-axis dim 4 matches.")

<a id="10"></a>
## Exercise 10 — Custom Op Shape Annotation

When using a custom operator, shape inference has no built-in rule.
Manually annotate the graph's `value_info` to provide shape information
so that downstream nodes can be inferred.

In [ ]:
# Custom op: my.domain::SwishActivation
# Semantics: element-wise, output shape = input shape

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 512])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)
Z = helper.make_tensor_value_info("Z", TensorProto.FLOAT, None)

W_np = np.zeros((512, 10), dtype=np.float32)
W = helper.make_tensor("W", TensorProto.FLOAT, [512, 10], W_np.flatten())

nodes = [
    helper.make_node("SwishActivation", ["X"], ["Y"], domain="my.domain"),
    helper.make_node("MatMul", ["Y", "W"], ["Z"]),
]

graph = helper.make_graph(nodes, "custom_op", [X], [Z], initializer=[W])
model = helper.make_model(graph, opset_imports=[
    helper.make_opsetid("", 17),
    helper.make_opsetid("my.domain", 1),
])

# Before annotation
shapes_before = get_shapes(model)
print("Before annotation:")
print(f"  Y (custom op output): {shapes_before.get('Y', 'MISSING')}")
print(f"  Z (MatMul output):    {shapes_before.get('Z', 'MISSING')}")

# Annotate: SwishActivation is element-wise, so output = input shape
y_info = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 512])
model.graph.value_info.append(y_info)

shapes_after = get_shapes(model)
print("\nAfter annotation:")
print(f"  Y (custom op output): {shapes_after.get('Y', 'MISSING')}")
print(f"  Z (MatMul output):    {shapes_after.get('Z', 'MISSING')}")
print("\nAnnotation enabled shape inference for downstream MatMul!")

In [ ]:
# Summary: Shape inference cheat sheet
rules = [
    ("MatMul",    "[...,M,K] x [...,K,N] -> [...,M,N]",        "Inner dims must match"),
    ("Conv",      "floor((H+2p-d(k-1)-1)/s)+1",                "Spatial output formula"),
    ("Add/Mul",   "max(d_A, d_B) per dim, right-aligned",       "NumPy broadcasting"),
    ("Transpose", "shape(Y)[i] = shape(X)[perm[i]]",            "Attribute-based"),
    ("Concat",    "sum on axis dim, match on others",            "Axis dimension sums"),
    ("Reshape",   "product(in) = product(out), -1 inferred",    "Volume conservation"),
    ("Relu/Sig",  "output shape = input shape",                  "Element-wise identity"),
    ("NonZero",   "[rank, ?] — data-dependent",                  "Partial inference"),
]

print(f"{'Operator':<12s}  {'Shape Rule':<42s}  {'Note'}")
print("=" * 80)
for op, rule, note in rules:
    print(f"{op:<12s}  {rule:<42s}  {note}")

## Summary

In these exercises you:

1. Ran `shape_inference.infer_shapes()` on models and inspected the resulting `TypeProto`.
2. Verified the Conv output formula $H_{out} = \lfloor (H + 2p - d(k-1) - 1)/s \rfloor + 1$ across multiple configurations.
3. Tracked shape propagation through a multi-layer CNN.
4. Tested broadcasting rules for element-wise operators.
5. Propagated symbolic dimensions (`batch`, `seq`) through attention-style graphs.
6. Observed partial inference for data-dependent ops like `NonZero`.
7. Diagnosed and fixed shape mismatch errors.
8. Annotated custom op outputs to enable downstream inference.